## Challenging SRL models (B): **BERT**

In [1]:
import a2standalone
import a2calculation

D:\anaconda3\envs\ML4NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Below are the relevant files that we need for this notebook.

In [2]:
# Challenge datasets
a1_1= './Data/A1.1_MFT_ARG0_SD.tsv'
a1_2= './Data/A1.2_INV_ARG0_LD.tsv'
b1= './Data/B1_DIR_predicate_subst.tsv'
c1= './Data/C1_MFT_ARG1_Cognate_object.tsv'
d1= './Data/D1_INV_ARG1_CAU_INC.tsv'
d2= './Data/D2_INV_ARG1_Voice.tsv'
e1= './Data/E1_DIR_PP_TMP_LOC.tsv'
f1= './Data/F1_MFT_ARG0_Verb_order.tsv'

#prediction files
pa1_1= './Predictions/berta11.tsv'
pa1_2= './Predictions/berta12.tsv'
pb1= './Predictions/bertb1.tsv'
pc1= './Predictions/bertc1.tsv'
pd1= './Predictions/bertd1.tsv'
pd2= './Predictions/bertd2.tsv'
pe1= './Predictions/berte1.tsv'
pf1= './Predictions/bertf1.tsv'
#logreg and vectoriser
model= './Models/bertsrl'


### Testing/Predicting

The trained logistic regression model is then tested on the challenge datasets above. This is achieved through the function below.

In [3]:
def log_challenge (infile, model, prediction_file):
    '''
    The function loads the trained model and vectorizer to predict the challenge dataset. 
    Then it automatically calculates the failure rate.

    Assumption: The .tsv is in a four-column format:
    0) use_case separated by space
    1) predicate; token in focus
    2) predicate labels ('_'or'x') separated by commas
    3) argument labels (argument labels or 'O') separated by commas

    parameters:
    -infile: file path to the dataset
    -model: the trained LogReg
    -vectorizer: the trained  vectorizer
    -prediction_file: a tsv file to save features and predictions
    returns:
    gold: a list of lists
    pred: a list of preds
    '''
    with open (infile, 'r', encoding= 'utf-8') as f:
        lines= f.read().split('\n')

    
    god= []
    pred= []
    output_prediction= '_'
    with open(prediction_file, 'w', encoding='utf-8') as output:
    
        for line in lines:
            if line: #skip empty lines
                cols= line.strip().split('\t')
                tokens= cols[0].strip().strip('"').split(' ')
                predicate= cols[2].strip().strip('"').split(',')
                argument= cols[3].strip().strip('"').split(',')
    
                #using the standalone function to predict each sentence
                processed_sentence, prediction= a2standalone.predict(tokens, predicate, model)
                actual_pre= prediction[0]

                #output the updated prediction
                output_tokens= ' '.join(tokens)
                output_gold= ','.join(argument)
                output_prediction= ','.join(actual_pre)
                output.write( output_tokens + '\t'+ output_gold+ '\t'+ output_prediction +'\n')

                #collect gold and predictions for later use
                god.append(argument)
                pred.append(actual_pre)
    
    return god, pred

#### A1.1 and A1.2

In [4]:
a11gold, a11prd= log_challenge (a1_1, model, pa1_1)

D:\anaconda3\envs\ML4NLP\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [5]:
a2calculation.mft_error (a11gold, a11prd)

The MFT/general error rate is 0.12. Case(s) [13, 67, 68, 73, 78, 85, 86, 87, 93, 95, 98, 99] is/are wrong


In [6]:
a12gold, a12prd= log_challenge (a1_2, model, pa1_2)

In [7]:
a2calculation.inv_two_datasets (a11gold, a11prd, a12gold, a12prd)

The raw error rate is 0.06, with [39, 56, 77, 79, 88, 93] wrong
The conditional error rate (i.e. cases where starting sentences are correctly predicted) is 0.06, with [39, 56, 77, 79, 88] wrong


#### B1

In [4]:
b1gold, b1prd= log_challenge (b1, model, pb1)

D:\anaconda3\envs\ML4NLP\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [5]:
a2calculation.dir_onedataset (b1gold, b1prd)

The raw error rate is 0.71, with pair [1, 3, 4, 5, 6] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) ia 0.75, with pair [3, 5, 6] wrong


#### C1

In [10]:
c1gold, c1prd= log_challenge (c1, model, pc1)

In [11]:
a2calculation.mft_error (c1gold, c1prd)

The MFT/general error rate is 0.12. Case(s) [2] is/are wrong


#### D1, D2

In [12]:
d1gold, d1prd= log_challenge (d1, model, pd1)

In [13]:
a2calculation.inv_onedataset (d1gold, d1prd)

The raw error rate is 0.04, with pair [1, 48] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) is 0.04, with pair [1, 48] wrong


In [14]:
d2gold, d2prd= log_challenge (d2, model, pd2)

In [15]:
a2calculation.inv_onedataset (d2gold, d2prd)

The raw error rate is 0.02, with pair [21] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) is 0.0, with pair [] wrong


#### E1

In [16]:
e1gold, e1prd= log_challenge (e1, model, pe1)

In [17]:
a2calculation.dir_onedataset (e1gold, e1prd)

The raw error rate is 0.2, with pair [7, 8] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) ia 0.0, with pair [] wrong


#### F1

In [18]:
f1gold, f1prd= log_challenge (f1, model, pf1)

In [19]:
a2calculation.mft_error (f1gold, f1prd)

The MFT/general error rate is 0.88. Case(s) [1, 3, 4, 5, 6, 7, 8] is/are wrong
